In [1]:
import pandas as pd
df = pd.read_csv(r"C:\online_retail\data\raw\online_retail_II.csv")

In [2]:
# Count how many rows belong to each StockCode
stockcode_counts = df["StockCode"].value_counts()

print("Total unique StockCodes:", df["StockCode"].nunique())
print("\nTop 20 most frequent StockCodes:")
print(stockcode_counts.head(20))

Total unique StockCodes: 5305

Top 20 most frequent StockCodes:
StockCode
85123A    5829
22423     4424
85099B    4216
21212     3318
20725     3259
84879     2960
47566     2768
21232     2747
22197     2549
22383     2540
20727     2529
21931     2434
22386     2347
22469     2325
22411     2297
84991     2271
22382     2251
22384     2230
21080     2224
22086     2217
Name: count, dtype: int64


In [3]:
stockcode_counts.describe()

count    5305.000000
mean      201.200943
std       346.293886
min         1.000000
25%        17.000000
50%        75.000000
75%       232.000000
max      5829.000000
Name: count, dtype: float64

In [4]:
stockcode_counts.tail(20)

StockCode
46776a    1
46776b    1
46776e    1
46776f    1
47593b    1
51014c    1
84031a    1
84031b    1
90183c    1
90184c    1
23602     1
23664     1
23449     1
23652     1
72802c    1
85049c    1
84971l    1
85034b    1
23609     1
23617     1
Name: count, dtype: int64

In [5]:
# Show descriptions for the least frequent StockCodes
rare_stockcodes = stockcode_counts[
    stockcode_counts == 1
].index

rare_items = df[
    df["StockCode"].isin(rare_stockcodes)
][["StockCode", "Description", "Quantity", "Price", "Invoice"]]

rare_items.head(30)

,StockCode,Description,Quantity,Price,Invoice
470,21646,NaN,-50,0.00,489521
1898,20739,*USB Office Glitter Lamp,1,8.65,489597
2201,30086C,ASSORTED FRUIT STRAWS,1,1.70,489597
2534,72529w,WHITE ADVENT CANDLE,1,4.30,489600
2578,79070b,ENGLISH ROSE SQUARE CAKE TINS,1,11.26,489602
2695,85164b,BLACK BAROQUE CUCKOO CLOCK,1,26.04,489609
2795,35004s,SET OF 3 SILVER FLYING DUCKS,1,13.00,489614
3162,35956,lost,-1043,0.00,489660
3168,35605A,damages,-117,0.00,489663
5205,21860,DR WHO TOP TRUMPS GAME,1,5.17,489856


In [6]:
# Investigate unusual patterns in rare StockCode rows

rare_df = df[
    df["StockCode"].isin(rare_stockcodes)
].copy()

print("Total rows with StockCodes appearing once:",
      len(rare_df))

print("\nMissing descriptions:",
      rare_df["Description"].isna().sum())

print("\nZero prices:",
      rare_df["Price"].eq(0).sum())

print("\nNegative quantities:",
      rare_df["Quantity"].lt(0).sum())

print("\nMissing description AND zero price:",
      (rare_df["Description"].isna() &
       rare_df["Price"].eq(0)).sum())

print("\nNegative quantity AND zero price:",
      (rare_df["Quantity"].lt(0) &
       rare_df["Price"].eq(0)).sum())

Total rows with StockCodes appearing once: 433

Missing descriptions: 347

Zero prices: 358

Negative quantities: 304

Missing description AND zero price: 347

Negative quantity AND zero price: 301


In [7]:
# Compare missing descriptions across StockCode frequencies

missing_desc = df[df["Description"].isna()].copy()

print("Rows with missing descriptions:", len(missing_desc))

print("Unique StockCodes among missing descriptions:",
      missing_desc["StockCode"].nunique())

print("\nMost frequent StockCodes with missing descriptions:")
print(missing_desc["StockCode"].value_counts().head(20))

print("\nInvoice prefixes for rows with missing descriptions:")
print(missing_desc["Invoice"].astype(str).str[0].value_counts())

Rows with missing descriptions: 4382
Unique StockCodes among missing descriptions: 2451

Most frequent StockCodes with missing descriptions:
StockCode
22139     12
84990     12
79321     11
35965     11
22950     10
23084     10
22087      9
22084      9
71477      8
37461      8
72803B     7
37509      7
35970      7
84977      7
21768      7
22528      7
POST       7
84795D     7
22451      7
37446      6
Name: count, dtype: int64

Invoice prefixes for rows with missing descriptions:
Invoice
5    3334
4    1048
Name: count, dtype: int64


In [8]:
missing_desc = df[df["Description"].isna()].copy()

print("Price distribution:")
print(missing_desc["Price"].describe())

print("\nZero-price rows:",
      missing_desc["Price"].eq(0).sum())

print("\nNegative-price rows:",
      missing_desc["Price"].lt(0).sum())

print("\nQuantity distribution:")
print(missing_desc["Quantity"].describe())

print("\nNegative-quantity rows:",
      missing_desc["Quantity"].lt(0).sum())

print("\nPositive-quantity rows:",
      missing_desc["Quantity"].gt(0).sum())

Price distribution:
count    4382.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

Zero-price rows: 4382

Negative-price rows: 0

Quantity distribution:
count    4382.000000
mean      -17.262209
std       509.704959
min     -9600.000000
25%       -28.000000
50%        -4.000000
75%         3.000000
max      9600.000000
Name: Quantity, dtype: float64

Negative-quantity rows: 2689

Positive-quantity rows: 1693


In [9]:

missing_desc = df[df["Description"].isna()].copy()

print("Customer ID availability:")
print(missing_desc["Customer ID"].isna().value_counts())

print("\nInvoice prefix and quantity sign:")

missing_desc["InvoicePrefix"] = (
    missing_desc["Invoice"].astype(str).str[0]
)

missing_desc["QuantityType"] = missing_desc["Quantity"].apply(
    lambda x: "Negative" if x < 0
    else "Positive" if x > 0
    else "Zero"
)

print(pd.crosstab(
    missing_desc["InvoicePrefix"],
    missing_desc["QuantityType"],
    margins=True
))

Customer ID availability:
Customer ID
True    4382
Name: count, dtype: int64

Invoice prefix and quantity sign:
QuantityType   Negative  Positive   All
InvoicePrefix                          
4                   687       361  1048
5                  2002      1332  3334
All                2689      1693  4382


In [10]:
missing_desc = df[df["Description"].isna()].copy()

print("Top 20 StockCodes among missing descriptions:")
print(missing_desc["StockCode"].value_counts().head(20))

print("\nMost common StockCodes with their quantities and invoice prefixes:")
top_missing_codes = (
    missing_desc["StockCode"]
    .value_counts()
    .head(10)
    .index
)

print(
    missing_desc[
        missing_desc["StockCode"].isin(top_missing_codes)
    ][["StockCode", "Quantity", "Price", "Invoice"]]
    .sort_values("StockCode")
    .head(30)
)

Top 20 StockCodes among missing descriptions:
StockCode
22139     12
84990     12
79321     11
35965     11
22950     10
23084     10
22087      9
22084      9
71477      8
37461      8
72803B     7
37509      7
35970      7
84977      7
21768      7
22528      7
POST       7
84795D     7
22451      7
37446      6
Name: count, dtype: int64

Most common StockCodes with their quantities and invoice prefixes:
        StockCode  Quantity  Price Invoice
783735      22084         3    0.0  559604
759230      22084        44    0.0  557486
710628      22084       -56    0.0  552766
790201      22084        15    0.0  560116
824975      22084         1    0.0  563101
845362      22084         5    0.0  564922
873506      22084         7    0.0  567371
696953      22084      -150    0.0  551422
908127      22084         2    0.0  569929
413033      22087         1    0.0  528887
710630      22087       -26    0.0  552768
720798      22087         2    0.0  553717
429455      22087         3    

In [11]:
# Count unique descriptions associated with each StockCode
description_counts = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .nunique()
)

# StockCodes with missing descriptions in at least one row
missing_codes = df.loc[
    df["Description"].isna(), "StockCode"
].unique()

# Among these codes, how many have descriptions elsewhere?
codes_with_description = description_counts.reindex(
    missing_codes
).fillna(0)

print(
    "Missing-description StockCodes with a description elsewhere:",
    (codes_with_description > 0).sum()
)

print(
    "Missing-description StockCodes with no description elsewhere:",
    (codes_with_description == 0).sum()
)

Missing-description StockCodes with a description elsewhere: 2096
Missing-description StockCodes with no description elsewhere: 355


In [12]:

sc = df['StockCode'].astype(str).str.strip().str.upper()
df['SC_clean'] = sc

# Split into numeric base + letter suffix
parts = sc.str.extract(r'^(?P<base>\d{5})(?P<suffix>[A-Z]*)$')
df['sc_base'] = parts['base']
df['sc_suffix'] = parts['suffix']

# How many codes follow the "5 digits + suffix" pattern?
print("Matches pattern:", parts['base'].notna().mean())
print("Non-matching (POST, M, BANK CHARGES, etc.):")
print(df.loc[parts['base'].isna(), 'SC_clean'].value_counts().head(20))

# Bases that have multiple suffix variants
variants = (df.dropna(subset=['sc_base'])
              .groupby('sc_base')['SC_clean'].nunique())
print("Bases with >1 code:", (variants > 1).sum(), "of", len(variants))

# For those, are descriptions identical or different?
multi = variants[variants > 1].index
sub = df[df['sc_base'].isin(multi)].dropna(subset=['Description'])
g = sub.groupby('sc_base').agg(
    n_codes=('SC_clean', 'nunique'),
    n_desc=('Description', 'nunique'),
    n_prices=('Price', 'nunique'),
)
print("Same description across codes:", (g['n_desc'] == 1).sum())
print("Different descriptions:", (g['n_desc'] > 1).sum())

# Eyeball a few
print(sub[sub['sc_base'] == '85183']
      .groupby(['SC_clean', 'Description'])['Price'].agg(['count', 'median']))

Matches pattern: 0.9942915818398663
Non-matching (POST, M, BANK CHARGES, etc.):
SC_clean
POST            2122
DOT             1446
M               1426
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
GIFT_0001_20      29
GIFT_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
GIFT_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
Name: count, dtype: int64
Bases with >1 code: 430 of 4198
Same description across codes: 31
Different descriptions: 390
                                              count  median
SC_clean Description                                       
85183A   CHARLIE & LOLA WASTEPAPER BIN BLUE     147    1.25
85183B   CHARLIE & LOLA WASTEPAPER BIN FLORA    178    1.25
